# DIT5411 — Traditional Chinese Handwriting Recognition (Kaggle)

**Before running:** Settings → Accelerator → **GPU** (P100), and Settings → Internet → **On**.

Kaggle gives 30 GPU-hours/week (vs Colab's ~2h/day). The dataset is downloaded inside the notebook — no upload needed. Checkpoints go to `/kaggle/working/results/checkpoints`; use **Save Version** periodically to preserve them as Output, then **"Copy & Run"** from the saved version to resume (`--resume`).

In [ ]:
import os, sys, subprocess, glob

# Clone the project code (public repo)
if not os.path.isfile('train.py'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/cveg1234/DIT5411-LuiYuHin.git'], check=True)
    os.chdir('DIT5411-LuiYuHin')
subprocess.run(['git', 'pull', '-q'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

import tensorflow as tf
print('Working dir:', os.getcwd())
print('GPU:', tf.config.list_physical_devices('GPU'))

## Download & extract the dataset (one time, ~15 min)
684,677 images across 13,065 characters. If `data/cleaned_data` already exists (e.g. from a saved Output), this cell skips.

In [ ]:
DATA_DIR = os.path.join(os.getcwd(), 'data', 'cleaned_data')
if not os.path.isdir(DATA_DIR) or len(os.listdir(DATA_DIR)) < 1000:
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/chenkenanalytic/handwritting_data_all.git'], check=True)
    with open('all_data.zip', 'wb') as out:
        for part in sorted(glob.glob('handwritting_data_all/all_data.zip.*')):
            with open(part, 'rb') as f:
                out.write(f.read())
    subprocess.run([sys.executable, 'extract_dataset.py',
                    '--zip', 'all_data.zip', '--dest', DATA_DIR], check=True)
print('Character folders:', len(os.listdir(DATA_DIR)))

## Quick pilot (optional — 20 classes, 2 epochs)

In [ ]:
r = subprocess.run([sys.executable, 'train.py', '--data-dir', DATA_DIR,
                    '--output-dir', 'results_pilot', '--num-classes', '20',
                    '--epochs', '2', '--models', 'A', 'B', '--no-imagenet',
                    '--steps-per-epoch', '30'])
print('pilot exit code:', r.returncode)
print(open('results_pilot/RESULTS.md', encoding='utf-8').read())

## Full assignment run
3 models × 5 epochs, full ~20,394 steps/epoch. At 128px on P100 this is roughly 22 GPU-hours — fits in Kaggle's 30h/week with time to spare.

**If interrupted:** click **Save Version** (saves checkpoints + dataset as Output), then later open that version and click **Copy & Run Code**; the cell below resumes from the last saved epoch via `--resume`.

In [ ]:
r = subprocess.run([sys.executable, 'train.py', '--data-dir', DATA_DIR,
                    '--output-dir', 'results', '--epochs', '5',
                    '--models', 'A', 'B', 'C', '--resume'])
print('full run exit code:', r.returncode)

In [ ]:
from IPython.display import Markdown, display
p = 'results/RESULTS.md'
display(Markdown(open(p, encoding='utf-8').read()) if os.path.isfile(p)
        else 'RESULTS.md not generated yet — finish training first.')